# Online Adaptive Forecasting

In [885]:
%load_ext autoreload
%autoreload 2

import re
import os
import json
import torch
print(torch.__version__)
print(torch.cuda.is_available())
import numpy as np
import matplotlib.pyplot as plt
import trajdata.visualization.vis as trajdata_vis
import pandas as pd

from torch.utils import data
from tqdm.notebook import tqdm
from trajectron.model.model_registrar import ModelRegistrar
from trajectron.model.model_utils import UpdateMode
from trajectron.model.trajectron import Trajectron
from collections import defaultdict
from pathlib import Path
from typing import DefaultDict, Dict, Final, List, Optional, Union
from trajdata import UnifiedDataset, AgentType, AgentBatch

fig_folder_name = 'test_figs'
seed = 0
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
1.13.1+cu117
True


In [886]:
# Change this to suit your computing environment and folder structure!

TRAJDATA_CACHE_DIR: Final[str] = "/home/mikolaj@acfr.usyd.edu.au/.unified_data_cache"
KITTI_SAMPLE_RAW_DATA_DIR: Final[str] = "/home/mikolaj@acfr.usyd.edu.au/datasets/KITTI"

In [1075]:
### TRO no map model: "mik_base_fullkitti_h0_2_p1_5-25_Oct_2024_14_25_56"
model_name = "0757_IV_03_base_fullkitti_est" #"0000_base_est"
base_model = "models/IV_models/" + model_name

tag = ""
if 'est' in model_name:
    tag = 'est'
elif 'sgt' in model_name:
    tag = 'sgt'
elif 'gt' in model_name:
    tag = 'gt'
else:
    raise Exception("wrong model name. Est methdo tag missing")

# k0_model = "models/nusc_mm_k0_tpp-12_Sep_2022_00_40_16"
#adaptive_model = "models/mik_base_fullkitti_h2_p1_5-16_Oct_2024_11_59_16"
# oracle_model = "models/lyft_mm_base_tpp-11_Sep_2022_18_56_49"

base_checkpoint = 20
# k0_checkpoint = 20
#adaptive_checkpoint = 20
# oracle_checkpoint = 1

eval_data = "visual_full_scene_kitti"

history_sec = 0.3
prediction_sec = 1.5

In [1076]:
AXHLINE_COLORS = {
    "Base": "#DD9787",
    "K0": "#A6C48A",
    "Oracle": "#A2999E"
}

SEABORN_PALETTE = {
    "Finetune": "#AA7C85",
    "K0": "#A6C48A",
    "Ours+Finetune": "#2D93AD",
    "Ours": "#9F9FED",
    "Base": "#DD9787",
    "K0+Finetune": "#67934D",
    "Oracle": "#A2999E"
}

In [1077]:
def load_model(model_dir: str, device: str, epoch: int = 10, custom_hyperparams: Optional[Dict] = None):
    save_path = Path(model_dir) / f'model_registrar-{epoch}.pt'

    model_registrar = ModelRegistrar(model_dir, device)
    with open(os.path.join(model_dir, 'config.json'), 'r') as config_json:
        hyperparams = json.load(config_json)
        
    if custom_hyperparams is not None:
        hyperparams.update(custom_hyperparams)

    trajectron = Trajectron(model_registrar, hyperparams, None, device)
    trajectron.set_environment()
    trajectron.set_annealing_params()

    checkpoint = torch.load(save_path, map_location=device)
    trajectron.load_state_dict(checkpoint["model_state_dict"], strict=False)

    return trajectron, hyperparams

In [1078]:
if torch.cuda.is_available():
    device = 'cuda:0'
    torch.cuda.set_device(0)
else:
    device = 'cpu'

In [1079]:
base_trajectron, hyperparams = load_model(
    base_model, device, epoch=base_checkpoint,
    custom_hyperparams={"trajdata_cache_dir": TRAJDATA_CACHE_DIR,
                        "single_mode_multi_sample": True}
)

In [1080]:
# Load training and evaluation environments and scenes
attention_radius = defaultdict(lambda: 20.0) # Default range is 20m unless otherwise specified.
attention_radius[(AgentType.PEDESTRIAN, AgentType.PEDESTRIAN)] = 10.0
attention_radius[(AgentType.PEDESTRIAN, AgentType.VEHICLE)] = 20.0
attention_radius[(AgentType.VEHICLE, AgentType.PEDESTRIAN)] = 20.0
attention_radius[(AgentType.VEHICLE, AgentType.VEHICLE)] = 30.0

# map_params = {"px_per_m": 2, "map_size_px": 100, "offset_frac_xy": (-0.75, 0.0)}

map_params = {"px_per_m": 1, "map_size_px": 100, "offset_frac_xy": (0.0, 0.0)}

online_eval_dataset = UnifiedDataset(
    desired_data=[eval_data],
    desired_dt=0.05,
    history_sec=(0.05, history_sec),
    future_sec=(prediction_sec, prediction_sec),
    agent_interaction_distances=attention_radius,
    incl_robot_future=hyperparams['incl_robot_node'],
    incl_raster_map=hyperparams['map_encoding'],
    raster_map_params=map_params, # None
    only_predict=[AgentType.VEHICLE],
    no_types=[AgentType.UNKNOWN],
    num_workers=0,
    # standardize_data=False,
    cache_location=TRAJDATA_CACHE_DIR,
    rebuild_cache=True,
    data_dirs={
        "visual_full_scene_kitti": KITTI_SAMPLE_RAW_DATA_DIR,
    },
    verbose=True
)

# Not used here, would be great to use it tho
# batch_eval_dataset = UnifiedDataset(
#     desired_data=[eval_data],
#     desired_dt=0.05,
#     history_sec=(history_sec, history_sec),
#     future_sec=(prediction_sec, prediction_sec),
#     agent_interaction_distances=attention_radius,
#     incl_robot_future=hyperparams['incl_robot_node'],
#     incl_raster_map=hyperparams['map_encoding'],
#     raster_map_params=None, # map_params,
#     only_predict=[AgentType.VEHICLE],
#     no_types=[AgentType.UNKNOWN],
#     num_workers=0,
#     cache_location=TRAJDATA_CACHE_DIR,
#     data_dirs={
#         "full_scene_kitti": KITTI_SAMPLE_RAW_DATA_DIR,
#     },
#     verbose=True
# )

Loading data for matched scene tags: ['test-poland-visual_full_scene_kitti']
Loading visual_full_scene_kitti dataset...
_get_matching_scenes_from_obj FINISHED


Calculating Agent Data (Serially): 100%|█████████| 1/1 [00:00<00:00, 147.70it/s]


1 scenes in the scene index.


Structuring Agent Data Index: 100%|█████████████| 1/1 [00:00<00:00, 8224.13it/s]


In [1081]:
def get_dataloader(
    eval_dataset: UnifiedDataset,
    batch_size: int = 128,
    num_workers: int = 0,
    shuffle: bool = False
):
    return data.DataLoader(
        eval_dataset,
        collate_fn=eval_dataset.get_collate_fn(pad_format="right"),
        pin_memory=False if device == 'cpu' else True,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers
    )

In [1082]:
metrics_list = ["ml_ade", "ml_fde", "nll_mean", "min_ade_5", "min_ade_10"]

## PREDICTION TO CSV

In [1083]:
def maybe_makedirs(path_to_create):
    """This function will create a directory, unless it exists already,
    at which point the function will return.
    The exception handling is necessary as it prevents a race condition
    from occurring.
    Inputs:
        path_to_create - A string path to a directory you'd like created.
    """
    try:
        os.makedirs(path_to_create)
    except OSError:
        if not os.path.isdir(path_to_create):
            raise

from trajdata.data_structures.batch import AgentBatch


# Function to save standrdize data prediction into the CSV file
def save_as_csv(frame_id, batch:AgentBatch, batch_idx, prediction, pred_horizon, scene_name):
    agent_id = batch.agent_name[0]
    current_state = batch.curr_agent_state[batch_idx].numpy() ######## Check if there is a heading in this
    curr_x = current_state[0]
    curr_y = current_state[1]
    curr_heading = current_state[6]
    
    agent_hist = batch.agent_hist[batch_idx].cpu()
    hist_x = agent_hist[:, 0]
    hist_y = agent_hist[:, 1]

    pred_x = prediction[batch_idx, 0:pred_horizon, 0]
    pred_y = prediction[batch_idx, 0:pred_horizon, 1]

    # Save CSV file:
    csv_output_path = os.path.join('/home/mikolaj@acfr.usyd.edu.au/datasets/KITTI/Jesse_processed/', scene_name, 'data', 'predictions_local', tag, str(agent_id))
    maybe_makedirs(csv_output_path)

    csv_path = os.path.join(csv_output_path, f'{frame_id}.csv')
    data = []
    
    # Append historical data (includes current state)
    for x, y in zip(hist_x, hist_y):
        data.append(['History', x.item(), y.item()])
        
    # Append future predictions
    for x, y in zip(pred_x, pred_y):
        data.append(['Future', x, y])
        
    # Create a DataFrame and save to CSV
    df_traj = pd.DataFrame(data, columns=['Type', 'X', 'Y'])
    df_traj['obj_id'] = agent_id
    df_traj['frame_id'] = frame_id + pred_horizon - (len(df_traj) - 1 - df_traj.index) # HERE is ok I guess
    df_traj.to_csv(csv_path, index=False)


    # De-Standardize it:
    scene_path   = os.path.join("/home/mikolaj@acfr.usyd.edu.au/datasets/KITTI/Jesse_processed", scene_name, 'data', 'object_pose_motion.csv')
    df_scene = pd.read_csv(scene_path)
    # df_curr_frame = df_scene[(df_scene['object_id'] == agent_id) & (df_scene['frame_id'] == frame_id)]
    # old_x = df_curr_frame['x'].iloc[0]
    # old_y = df_curr_frame['y'].iloc[0]
    # old_heading = df_curr_frame['heading'].iloc[0]

    original_x = []
    original_y = []

    # Loop through the DataFrame to "de-standardize" values
    for index, row in df_traj.iterrows():
        x_standardized = row['X']
        y_standardized = row['Y']
    
        # Calculate original coordinates
        x_original = (x_standardized * np.cos(curr_heading) -
                      y_standardized * np.sin(curr_heading) + curr_x)
        y_original = (x_standardized * np.sin(curr_heading) +
                      y_standardized * np.cos(curr_heading) + curr_y)
    
        # Append to lists
        original_x.append(x_original)
        original_y.append(y_original)
        
    # Add the original coordinates back to a new DataFrame
    df_global = pd.DataFrame({
        'Type': df_traj['Type'],
        'x': original_x,
        'y': original_y,
    })

    df_global['object_id'] = agent_id
    df_global['frame_id'] = df_traj['frame_id']

    column_list = []
    if tag == 'est':
        column_list = ['object_id', 'frame_id', 'x', 'y', 'heading']
    elif tag == 'sgt':
        column_list = ['object_id', 'frame_id', 'sgt_x', 'sgt_y', 'sgt_heading']
    elif tag == 'gt':
        column_list = ['object_id', 'frame_id', 'gt_x', 'gt_y', 'gt_heading']
    # Merge the DataFrames on 'object_id' and 'frame_id'
    df_merged = pd.merge(df_global, df_scene[column_list], 
                         on=['object_id', 'frame_id'], 
                         how='left',
                         suffixes=('', '_actual'))

    df_merged = df_merged.rename(columns={
    'gt_x': 'x_actual', 
    'gt_y': 'y_actual',
    'gt_heading': 'heading_actual',
    'sgt_x': 'x_actual',
    'sgt_y': 'y_actual',
    'sgt_heading': 'heading_actual',
    'x_actual': 'x_actual',
    'y_actual': 'y_actual',
    'heading': 'heading_actual',
})
    
    csv_output_path = os.path.join('/home/mikolaj@acfr.usyd.edu.au/datasets/KITTI/Jesse_processed/', scene_name, 'data', 'predictions_global', tag, str(agent_id))
    maybe_makedirs(csv_output_path)

    csv_path = os.path.join(csv_output_path, f'{frame_id}.csv')
    df_merged.to_csv(csv_path, index=False)
    


## Online (Per Agent) Plotting

In [1084]:
import matplotlib.patches as patches
from scipy import linalg

import time

time_taken_for_all = 0
prog = re.compile("(.*)/(?P<scene_name>.*)/(.*)$")

def plot_outputs(
    eval_dataset: UnifiedDataset,
    dataset_idx: int,
    model: Trajectron,
    model_name: str,
    agent_ts: int,
    save=True,
    save_csv=True,
    extra_str=None,
    subfolder="",
    filetype="png",
    uncertainty=True,
):
    num_modes = 5
    plot_every = 1
    plot_square_radius = 20
    pred_horizon = 30
    batch: AgentBatch = eval_dataset.get_collate_fn(pad_format="right")([eval_dataset[dataset_idx]])
    batch_idx = 0

    fig, ax = plt.subplots(dpi=150)
    # NOTE: To exactly reproduce the figure from our paper, you must edit the trajdata_vis.plot_agent_batch function
    # to accept an optional future_horizon (int) argument which truncates plotting to only the future future_horizon timesteps
    # (otherwise it can be a but hard to read the figure when full-length predictions and futures are plotted).
    # After making the above change, please add future_horizon=pred_horizon to the plot_agent_batch arguments.
    trajdata_vis.plot_agent_batch(batch, batch_idx=batch_idx, ax=ax, show=False, close=False)

    scene_info_path, _, scene_ts = eval_dataset._data_index[dataset_idx]
    scene_name = prog.match(scene_info_path).group("scene_name")
    
    agent_name = batch.agent_name[0]
    agent_type_name = f"{str(AgentType(batch.agent_type[0].item()))}/{agent_name}"
    print(dataset_idx, scene_name, scene_ts, agent_type_name)

    with torch.no_grad():
        start_time = time.time()
        
        predictions = model.predict(batch,
                                    z_mode=True,
                                    gmm_mode=True,
                                    full_dist=False,
                                    output_dists=False)
        end_time = time.time()
        print(f"Time taken: {end_time - start_time}")
        global time_taken_for_all
        time_taken_for_all += (end_time - start_time)
        prediction = next(iter(predictions.values()))

        if uncertainty:
            prediction_distribution_dict, _ = model.predict(batch,
                                          z_mode=False,
                                          gmm_mode=False,
                                          full_dist=True,
                                          output_dists=True)
            pred_dist = next(iter(prediction_distribution_dict.values()))
    
            pi_threshold = 0.0
    
    batch.to("cpu")
        
    if uncertainty and pred_dist.mus.shape[:2] != (1, 1):
        return

    if uncertainty:
        means = pred_dist.mus[batch_idx, 0].cpu().numpy()
        covs = pred_dist.get_covariance_matrix()[batch_idx, 0].cpu().numpy()
        pis = pred_dist.pis_cat_dist.probs[batch_idx, 0].cpu().numpy()

    ax.plot(
        prediction[batch_idx, 0:pred_horizon:plot_every, 0],
        prediction[batch_idx, 0:pred_horizon:plot_every, 1],
        color='#2D93AD',
        alpha=1,
        label="Most Likely"
    )
    
    if uncertainty:
        for z_val in range(min(num_modes, means.shape[1])):
            # All pis are the same across time.
            pi = pis[0, z_val]
            
            if pi < pi_threshold:
                continue
    
            color = '#9F9FED'
    
            alpha_val = pi
            ax.plot(means[0:pred_horizon:plot_every, z_val, 0], means[0:pred_horizon:plot_every, z_val, 1],
                        color=color,
                        alpha=alpha_val)
    
            for timestep in range(0, min(pred_horizon, means.shape[0]), plot_every):
                mean = means[timestep, z_val]
                covar = covs[timestep, z_val]
    
                v, w = linalg.eigh(covar)
                v = 2. * np.sqrt(2.) * np.sqrt(v)
                u = w[0] / linalg.norm(w[0])
    
                # Plot an ellipse to show the Gaussian component
                angle = np.arctan2(u[1], u[0])
                angle = 180. * angle / np.pi  # convert to degrees
                ell = patches.Ellipse(mean, v[0], v[1], 180. + angle,
                                        color=color)
                ell.set_edgecolor(None)
                ell.set_clip_box(ax.bbox)
                ell.set_alpha(alpha_val)
                ax.add_artist(ell)
    
    # batch_eval: Dict[str, torch.Tensor] = evaluation.compute_batch_statistics_pt(
    #     batch.agent_fut[..., :2],
    #     prediction_output_dict=torch.from_numpy(prediction),
    #     y_dists=pred_dist
    # )
    
    ax.set_title(None)
    # ax.set_title(f"{scene_name}/t={scene_ts} {agent_type_name}")
    offset = 8
    ax.set_xlim((-plot_square_radius+offset, plot_square_radius+offset))
    ax.set_ylim((-plot_square_radius, plot_square_radius))
    # ax.set_aspect('auto')
    
    #ax.axis('off')                             # If you want x, y coords
    # ax.set_xticklabels([])
    # ax.set_yticklabels([])
    # ax.set_xlabel(None)
    # ax.set_ylabel(None)
    
    ax.legend(bbox_to_anchor=(1.04, 0.5), loc="best", borderaxespad=0, frameon=False)
    # print(model_name, extra_str, batch_eval)

    if save_csv:
        save_as_csv(agent_ts, batch, batch_idx, prediction, pred_horizon, scene_name)
    
    if save:
        fname = f"{fig_folder_name}/{subfolder}Base_{scene_name}_{agent_name}_frame{agent_ts}"
        if extra_str:
            fname += "_" + extra_str
        fig.savefig(fname + f".{filetype}", bbox_inches="tight")
        
        plt.close(fig)
    plt.close(fig)

In [1085]:
def per_agent_plot(
    model: Trajectron,
    model_name: str,
    batch: AgentBatch,
    agent_ts: int,
    plot=True,
):
    with torch.no_grad():
        if plot:
            plot_outputs(online_eval_dataset,
                         dataset_idx=batch.data_idx[0].item(),
                         model=model,
                         model_name=model_name,
                         agent_ts=agent_ts,
                         save=False,
                         save_csv=True)

In [1086]:
online_eval_dataloader = get_dataloader(online_eval_dataset, batch_size=1, shuffle=False)

#adaptive_trajectron.reset_adaptive_info()

N_SAMPLES = 1001

outer_pbar = tqdm(
    online_eval_dataloader,
    total=min(N_SAMPLES, len(online_eval_dataloader)),
    desc=f'Adaptive Eval PH={prediction_sec}',
    position=0,
)

plot_per_step = True

curr_agent: str = None
agent_ts: int = 0
online_batch: AgentBatch
for data_sample, online_batch in enumerate(outer_pbar):
    if data_sample >= N_SAMPLES:
        outer_pbar.close()
        break
            
    if online_batch.agent_name[0] != curr_agent:
        # Resetting the K_n, L_n for each Bayesian last layer.
        #adaptive_trajectron.reset_adaptive_info()
        
        # # Resetting the finetune baseline to its base.
        # finetune_trajectron, _ = load_model(
        #     base_model, device, epoch=base_checkpoint,
        #     custom_hyperparams={"trajdata_cache_dir": "/home/bivanovic/.unified_data_cache",
        #                         "single_mode_multi_sample": False}
        # )
        # k0_finetune_trajectron, _ = load_model(
        #     k0_model, device, epoch=k0_checkpoint,
        #     custom_hyperparams={"trajdata_cache_dir": "/home/bivanovic/.unified_data_cache",
        #                         "single_mode_multi_sample": False}
        # )

        curr_agent = online_batch.agent_name[0]
        agent_ts: int = 0
        
    with torch.no_grad():
        # This is the inference call that internally updates L_n and K_n.
        # adaptive_trajectron.adaptive_predict(
        #     online_batch,
        #     update_mode=UpdateMode.ITERATIVE
        # )
        base_trajectron.predict(
            online_batch,
            update_mode=UpdateMode.ITERATIVE
        )
    
    # finetune_update(finetune_trajectron, online_batch)
    # finetune_last_layer_update(k0_finetune_trajectron, online_batch)
    
    # # This is effectively measuring number of updates/observed data points.
    # agent_ts += 1
    frame_num = online_batch.scene_ts[0].item() ############## This is the problem. Need to set up to the + min frame (the ts is +1). This is a scene_ts. So I just need to make all df to start from frame 0
    if agent_ts % 1 == 0:
        per_agent_plot(base_trajectron, "Ours", online_batch, frame_num, plot=plot_per_step)
            
    # This is effectively measuring the most-recently seen timestep.
    agent_ts += 1
#print(f"Avg time taken:{time_taken_for_all/88}")

Adaptive Eval PH=1.5:   0%|          | 0/54 [00:00<?, ?it/s]

0 0757 82 AgentType.VEHICLE/18a
Time taken: 0.014014959335327148
1 0757 83 AgentType.VEHICLE/18a
Time taken: 0.014223337173461914
2 0757 84 AgentType.VEHICLE/18a
Time taken: 0.015665054321289062
3 0757 85 AgentType.VEHICLE/18a
Time taken: 0.01288914680480957
4 0757 86 AgentType.VEHICLE/18a
Time taken: 0.012636184692382812
5 0757 87 AgentType.VEHICLE/18a
Time taken: 0.013445377349853516
6 0757 88 AgentType.VEHICLE/18a
Time taken: 0.015487909317016602
7 0757 89 AgentType.VEHICLE/18a
Time taken: 0.014425992965698242
8 0757 90 AgentType.VEHICLE/18a
Time taken: 0.013901472091674805
9 0757 91 AgentType.VEHICLE/18a
Time taken: 0.013817548751831055
10 0757 92 AgentType.VEHICLE/18a
Time taken: 0.014168977737426758
11 0757 93 AgentType.VEHICLE/18a
Time taken: 0.013758420944213867
12 0757 94 AgentType.VEHICLE/18a
Time taken: 0.01466512680053711
13 0757 95 AgentType.VEHICLE/18a
Time taken: 0.01322031021118164
14 0757 96 AgentType.VEHICLE/18a
Time taken: 0.014048576354980469
15 0757 97 AgentType.VE